# Laboratório 02 — Busca não informada em um labirinto

**Fundamentos de Inteligência Artificial** · 2026.2 · Prof. Leonardo Garcia Marques

---

Neste laboratório você vai traduzir para Python o **laço genérico de busca** que
está no quadro:

> 1. Coloque o nó inicial na fronteira.
> 2. Enquanto a fronteira não estiver vazia:
>    1. **Retire** um nó da fronteira.
>    2. Se ele satisfaz o teste de objetivo, **devolva** o caminho.
>    3. Marque o estado como visitado.
>    4. **Expanda**: coloque na fronteira os sucessores ainda não visitados.
> 3. Se a fronteira esvaziou, não há solução.

**BFS e DFS são o mesmo código.** Muda apenas o passo 2.1 — de que ponto da
fronteira se retira o próximo nó.

> ⚙️ **Sem bibliotecas externas.** Só `collections.deque`, que já vem com o Python.
> Roda no Jupyter, no Colab ou em um terminal.



## Parte 0 — O labirinto

Já está pronto. Rode e veja o mapa: `S` é a entrada, `G` é a saída, `#` são paredes.


In [ ]:
LABIRINTO = [
    "S...#...##",
    ".##....##.",
    "......#...",
    ".#........",
    ".##.......",
    "#.#..#....",
    "...#....#.",
    "..####....",
    "..#..#..#.",
    ".#..###.#G",
]

# '#' é parede, '.' é livre, 'S' é a entrada e 'G' é a saída.
LINHAS, COLUNAS = len(LABIRINTO), len(LABIRINTO[0])

def acha(simbolo):
    for i, linha in enumerate(LABIRINTO):
        j = linha.find(simbolo)
        if j >= 0:
            return (i, j)

INICIO   = acha('S')   # estado inicial
OBJETIVO = acha('G')   # teste de objetivo: estado == OBJETIVO

def livre(i, j):
    return 0 <= i < LINHAS and 0 <= j < COLUNAS and LABIRINTO[i][j] != '#'

def desenha(caminho=None):
    marcas = set(caminho or [])
    for i, linha in enumerate(LABIRINTO):
        saida = ''
        for j, c in enumerate(linha):
            if (i, j) in marcas and c not in 'SG':
                saida += 'o'
            else:
                saida += c
        print(saida)

print('Início:', INICIO, ' Objetivo:', OBJETIVO)
desenha()


## Parte 1 — Ações e modelo de transição

Na formulação que fizemos em aula, as **ações** são os quatro movimentos e o
**modelo de transição** diz em que célula cada movimento nos deixa.

Complete a função abaixo.


In [ ]:
def vizinhos(estado):
    """Devolve a lista de estados alcançáveis a partir de `estado`."""
    i, j = estado
    resultado = []
    # TODO 1: para cada uma das quatro direções (cima, baixo, esquerda, direita),
    #         calcule a célula vizinha e, se ela estiver livre, adicione a `resultado`.
    #         Dica: as direções podem ser escritas como (-1, 0), (1, 0), (0, -1), (0, 1).
    return resultado


In [ ]:
# teste rápido — o início deve ter 2 vizinhos neste labirinto
print('vizinhos de', INICIO, '->', vizinhos(INICIO))
assert len(vizinhos(INICIO)) == 2, 'confira as quatro direções e o teste livre()'
print('ok')


## Parte 2 — A busca

Agora o laço genérico. Repare que ele **não sabe** se é BFS ou DFS: quem decide é
o parâmetro `estrategia`, usado no momento de retirar o nó da fronteira.

O dicionário `veio_de` guarda, para cada estado, de onde chegamos nele. É com ele
que reconstruímos o caminho no final.


In [ ]:
from collections import deque

def reconstroi(veio_de, estado):
    """Sobe do objetivo até o início usando o dicionário veio_de."""
    caminho = []
    while estado is not None:
        caminho.append(estado)
        estado = veio_de[estado]
    return caminho[::-1]

def busca(estrategia):
    """estrategia: 'bfs' ou 'dfs'. Devolve (caminho, nós expandidos)."""
    fronteira = deque([INICIO])
    veio_de   = {INICIO: None}
    visitados = set()
    expandidos = 0

    while fronteira:
        # TODO 2: retire um nó da fronteira.
        #   BFS -> o que entrou PRIMEIRO   (fila:  fronteira.popleft())
        #   DFS -> o que entrou POR ÚLTIMO (pilha: fronteira.pop())
        estado = None   # <-- substitua

        if estado in visitados:
            continue
        visitados.add(estado)
        expandidos += 1

        # TODO 3: teste de objetivo. Se `estado` for o OBJETIVO, devolva o caminho.
        #         Use a função reconstroi(veio_de, estado) logo abaixo.

        # TODO 4: expanda. Para cada vizinho ainda não visitado, registre de onde
        #         ele veio (veio_de) e coloque-o na fronteira.

    return None, expandidos   # fronteira esvaziou: não há solução


## Parte 3 — Rodar e comparar

Se tudo estiver certo, a BFS devolve um caminho **mais curto** que o da DFS.


In [ ]:
for estrategia in ['bfs', 'dfs']:
    caminho, expandidos = busca(estrategia)
    if caminho is None:
        print(estrategia.upper(), '-> não encontrou caminho')
        continue
    print(f'{estrategia.upper():4s} -> {len(caminho):3d} células no caminho '
          f'| {expandidos:3d} nós expandidos')
    desenha(caminho)
    print()


## Parte 4 — Responda no próprio notebook

1. Qual das duas estratégias devolveu o caminho mais curto? Isso era garantido ou
   foi sorte? Justifique com o critério de retirada da fronteira.
2. Qual delas expandiu **menos** nós? Por que isso não a torna melhor?
3. Comente a linha `if v not in visitados` da expansão: o que acontece com a DFS
   se você removê-la? Teste — mas salve o notebook antes.
4. Troque a ordem das quatro direções na função `vizinhos`. O caminho da BFS mudou?
   E o da DFS? Explique a diferença.


In [ ]:
# Espaço para experimentar as respostas 3 e 4


---

## Desafio (opcional) — o 8-puzzle

O laço de busca que você escreveu **não sabe** que está resolvendo um labirinto.
Ele só conhece `INICIO`, `OBJETIVO` e `vizinhos`.

Então: troque a formulação e o mesmo código resolve outro problema.

- **Estado:** uma tupla de 9 posições, com `0` representando o espaço vazio.
- **Ações:** mover o vazio para cima, baixo, esquerda ou direita.
- **Objetivo:** `(1, 2, 3, 4, 5, 6, 7, 8, 0)`.

Comece por um estado embaralhado a poucos movimentos do objetivo — o espaço de
estados do 8-puzzle tem 181.440 configurações alcançáveis, e a DFS se perde nele.


In [ ]:
# Desafio: sua formulação do 8-puzzle
